### Faires Füllen – Lösung Schritt für Schritt

*33. Bundeswettbewerb Informatik (2014), 1. Runde, Aufgabe 1*

In einem Jahrhunderte alten Rätsel treffen sich zwei Personen. Die erste hat einen vollen Behälter mit acht Maßen Wein bei sich, die andere zwei leere Behälter mit einem Fassungsvermögen von drei und fünf Maßen. Können sie den Wein ohne Zuhilfenahme von anderen Behältnissen so umfüllen, dass jeder vier Maße in seinen Behältern hat? Ja, aber ganz einfach findet man die Lösung nicht.

Wir wollen dieses Rätsel hier verallgemeinern und mit Computerhilfe lösen lassen. Wieder treffen sich zwei Personen. Jede trägt eine bestimmte Menge von Behältern mit sich. Es ist bekannt, wie groß und wie voll diese Behälter zunächst sind.

#### Aufgabe 1

Schreibe ein Programm, das diese Daten einliest und dann feststellt, ob ein Umfüllen so möglich ist, dass beide Personen die gleiche Gesamtmenge Weins in ihren Behältern vorfinden. Wenn ja, dann soll das Programm einen übersichtlichen Plan ausgeben, wie das Umfüllen in einer minimalen Anzahl von Schritten vollführt werden kann. Ein Schritt ist dabei das Gießen des Weins von einem Behälter A in einen Behälter B, bis entweder A leer oder B voll ist.

Wende dein Programm auf das ursprüngliche Rätsel an sowie auf die Beispiele, die du unter bundeswettbewerb-informatik.de findest.

**Zu den Beispieldaten:** `beispieldaten/fair-beispiel1.txt` bis `fair-beispiel3.txt`. Es gibt keine Formatbeschreibung. Aus den Dateien abgelesen: Erste Zeile die Zahl der Behälter der ersten Person, zweite Zeile die Fassungsvermögen aller Behälter (erst die der ersten Person, dann die der zweiten), dritte Zeile die Füllstände. `fair-beispiel1.txt` ist das ursprüngliche Rätsel (`1` / `8 5 3` / `8 0 0`).

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Zustände und Übergänge:** Bei Rätseln wie diesem beschreibt man die Lage durch einen **Zustand** (hier: wie viel Wein in jedem Behälter ist) und alle erlaubten Züge durch **Übergänge** von einem Zustand zu einem anderen. Alle Zustände mit ihren Übergängen bilden einen **Graphen**.
- **Breitensuche:** Man besucht zuerst alle Zustände, die man mit 1 Zug erreicht, dann alle mit 2 Zügen, und so weiter. Der erste Zustand, der das Ziel erfüllt, wurde also mit der **kleinsten Zahl von Zügen** erreicht. Das ist genau die „minimale Anzahl von Schritten“ der Aufgabe.
- **Warteschlange (Queue):** Dafür braucht man eine Liste, in die man hinten einfügt und aus der man vorne entnimmt.
- **Vorgänger merken:** Für jeden besuchten Zustand speichert man, von welchem Zustand und mit welchem Zug man ihn zuerst erreicht hat. So kann man den Weg am Ende rückwärts nachlaufen und den Plan ausgeben. Das Wörterbuch sagt außerdem, ob ein Zustand schon besucht ist.

**Nützliche Funktionen und Bibliotheken**
- `collections.deque`: eine Warteschlange mit `append` (hinten einfügen) und `popleft` (vorne entnehmen). Bei einer normalen Liste wäre `pop(0)` bei langen Listen langsam.
- `tuple(liste)`: Listen können nicht als Schlüssel eines Wörterbuchs dienen, Tupel schon. Ein Zustand wird deshalb als Tupel gespeichert.
- `min(a, b)`, `sum(liste)`, `liste.reverse()`.

## Die Idee in Worten

**Modell.** Es gibt `k` Behälter. Die ersten `m` gehören der ersten Person, die übrigen der zweiten. Ein **Zustand** ist das Tupel der Füllstände `(f_1, ..., f_k)`. Ein **Zug** gießt von Behälter `a` in Behälter `b` (`a ≠ b`), wenn `a` nicht leer und `b` nicht voll ist. Dabei fließt `min(f_a, G_b - f_b)` Wein, wobei `G_b` das Fassungsvermögen von `b` ist: bis `a` leer oder `b` voll ist.

**Ziel.** Die Gesamtmenge `S` ändert sich nie. Gesucht ist ein Zustand, in dem die Behälter der ersten Person zusammen genau `S / 2` enthalten. Ist `S` ungerade, geht es sofort nicht.

**Verfahren.** Breitensuche ab dem Startzustand, bis ein Zielzustand erreicht ist. Wird die Warteschlange leer, ohne dass ein Zielzustand vorkam, gibt es **keine Lösung**.

Das Verfahren ist richtig, weil jeder Zustand, den man mit `d` Zügen erreichen kann, spätestens in „Runde `d`“ der Breitensuche in der Warteschlange landet. Und wir besuchen jeden Zustand nur einmal, denn ein zweiter, späterer Weg zu ihm wäre nie kürzer.

## Schritt 1: Daten einlesen

Die erste Zahl ist die Zahl `m` der Behälter der ersten Person. Die zweite Zeile enthält die Fassungsvermögen, die dritte die Füllstände. Wir lesen die Datei zeilenweise, wandeln jede Zeile in eine Liste von Zahlen um und geben `m`, die Fassungsvermögen und die Füllstände zurück.

In [ ]:
def lies(pfad):
    """Liest m, die Fassungsvermögen und die Füllstände."""
    with open(pfad) as datei:
        zeilen = datei.read().strip().split('\n')
    m = int(zeilen[0])
    groesse = [int(wort) for wort in zeilen[1].split()]
    fuell = [int(wort) for wort in zeilen[2].split()]
    return m, groesse, fuell


m, groesse, fuell = lies('beispieldaten/fair-beispiel1.txt')
print(m, groesse, fuell)

1 [8, 5, 3] [8, 0, 0]


## Schritt 2: Ziel prüfen und Züge bestimmen

Zuerst zwei kleine Funktionen:

- `ist_ziel(zustand, m, gesamt)`: Die Behälter `0` bis `m - 1` gehören der ersten Person. Ihre Summe muss genau die Hälfte der Gesamtmenge sein. Wir vergleichen `2 * summe` mit `gesamt`, dann braucht man keine Kommazahlen und ein ungerades `gesamt` fällt von selbst durch.
- `folgezustaende(zustand, groesse)`: Probiert alle Paare `(a, b)` mit `a ≠ b`. Ist `a` nicht leer und `b` nicht voll, gießt man `min(zustand[a], groesse[b] - zustand[b])` um. Die Funktion gibt eine Liste von Paaren `(neuer Zustand, (a, b))` zurück, damit wir später den Zug für den Plan wissen.

Behälter werden ab 0 gezählt, im Plan geben wir sie später ab 1 aus.

In [ ]:
def ist_ziel(zustand, m, gesamt):
    return 2 * sum(zustand[:m]) == gesamt


def folgezustaende(zustand, groesse):
    ergebnis = []
    for a in range(len(zustand)):
        for b in range(len(zustand)):
            if a != b and zustand[a] > 0 and zustand[b] < groesse[b]:
                menge = min(zustand[a], groesse[b] - zustand[b])
                neu = list(zustand)
                neu[a] = neu[a] - menge
                neu[b] = neu[b] + menge
                ergebnis.append((tuple(neu), (a, b)))
    return ergebnis


for neu, zug in folgezustaende((8, 0, 0), [8, 5, 3]):
    print(zug, '->', neu)

(0, 1) -> (3, 5, 0)
(0, 2) -> (5, 0, 3)


Vom Startzustand `(8, 0, 0)` gibt es nur zwei Züge: Der erste Behälter kann in den zweiten (5 Maß, dann ist dieser voll) oder in den dritten Behälter (3 Maß) gegossen werden. Das passt zu Abbildung 1 der Lösungshinweise.

## Schritt 3: Breitensuche

Die Datenstrukturen:

- `vorgaenger`: Wörterbuch. Schlüssel ist ein besuchter Zustand, Wert ist ein Paar `(Zustand davor, Zug)`. Für den Startzustand steht `None`. Das Wörterbuch beantwortet auch die Frage „schon besucht?“ (`zustand in vorgaenger`).
- `warteschlange`: Die Zustände, deren Folgezustände wir noch nicht bestimmt haben.

Ablauf: Nimm den vordersten Zustand aus der Warteschlange. Ist er ein Ziel, bist du fertig und gehst den Weg über `vorgaenger` rückwärts bis zum Start, danach wird die Liste umgedreht. Sonst: Jeden Folgezustand, der noch nicht besucht ist, trägst du in `vorgaenger` ein und hängst ihn hinten an die Warteschlange. Wird die Warteschlange leer, gibt es keine Lösung.

Die Funktion gibt die Liste der Züge als Paare `(Zustand, Zug)` zurück, jeweils den Zustand **nach** dem Zug, und `None`, wenn es keine Lösung gibt. Zusätzlich zählen wir die besuchten Zustände.

In [ ]:
from collections import deque


def suche(m, groesse, fuell):
    """Breitensuche. Gibt (Plan, Zahl besuchter Zustände) zurück, Plan = None, wenn es keine Lösung gibt."""
    gesamt = sum(fuell)
    start = tuple(fuell)
    vorgaenger = {start: None}
    warteschlange = deque([start])
    while len(warteschlange) > 0:
        zustand = warteschlange.popleft()
        if ist_ziel(zustand, m, gesamt):
            plan = []
            while vorgaenger[zustand] is not None:
                davor, zug = vorgaenger[zustand]
                plan.append((zustand, zug))
                zustand = davor
            plan.reverse()
            return plan, len(vorgaenger)
        for neu, zug in folgezustaende(zustand, groesse):
            if neu not in vorgaenger:
                vorgaenger[neu] = (zustand, zug)
                warteschlange.append(neu)
    return None, len(vorgaenger)

## Schritt 4: Plan übersichtlich ausgeben

Wir geben eine Tabelle aus: Der Kopf zeigt die Fassungsvermögen (die beiden Personen durch `|` getrennt), dann kommt die Startzeile und je eine Zeile pro Schritt mit dem Zug („Behälter 1 → 2“, gezählt ab 1) und den neuen Füllständen. Am Ende steht die Zahl der Schritte und, was jede Person hat.

In [ ]:
def zeile(werte, m):
    """Füllstände als Text, die beiden Personen durch | getrennt."""
    links = ' '.join(str(w).rjust(3) for w in werte[:m])
    rechts = ' '.join(str(w).rjust(3) for w in werte[m:])
    return links + ' |' + rechts.rjust(len(rechts) + 1)


def loese(pfad):
    m, groesse, fuell = lies(pfad)
    print('Behälter     ', zeile(groesse, m))
    print('Start        ', zeile(fuell, m))
    plan, besucht = suche(m, groesse, fuell)
    if plan is None:
        print('Keine Lösung (alle', besucht, 'erreichbaren Zustände geprüft).')
        return
    for nr, (zustand, (a, b)) in enumerate(plan, 1):
        print('Schritt', str(nr).rjust(2) + ':', str(a + 1) + ' -> ' + str(b + 1), ' ', zeile(zustand, m))
    print('Ziel nach', len(plan), 'Schritten: erste Person', sum(zustand[:m]), ', zweite Person', sum(zustand[m:]),
          '(' + str(besucht), 'Zustände besucht)')

## Schritt 5: Alle Beispiele

**Das ursprüngliche Rätsel** (8 Maß in einem Behälter, die zweite Person hat Behälter für 5 und 3 Maß):

In [ ]:
loese('beispieldaten/fair-beispiel1.txt')

Behälter        8 |   5   3
Start           8 |   0   0
Schritt  1: 1 -> 2     3 |   5   0
Schritt  2: 2 -> 3     3 |   2   3
Schritt  3: 3 -> 1     6 |   2   0
Schritt  4: 2 -> 3     6 |   0   2
Schritt  5: 1 -> 2     1 |   5   2
Schritt  6: 2 -> 3     1 |   4   3
Schritt  7: 3 -> 1     4 |   4   0
Ziel nach 7 Schritten: erste Person 4 , zweite Person 4 (16 Zustände besucht)


Der Plan stimmt mit den Lösungshinweisen überein (die Zustände `(8,0,0), (3,5,0), (3,2,3), (6,2,0), (6,0,2), (1,5,2), (1,4,3), (4,4,0)`, also **7 Schritte**).

Es gibt nur **16** erreichbare Zustände, so viele Knoten hat auch der Suchbaum in Abbildung 2 der Hinweise.

**Beispiel 2** (`10 8 | 11 7`, Füllstände `10 4 | 4 6`, zusammen 24, jede Person soll 12 haben):

In [ ]:
loese('beispieldaten/fair-beispiel2.txt')

Behälter       10   8 |  11   7
Start          10   4 |   4   6
Keine Lösung (alle 138 erreichbaren Zustände geprüft).


Hier gibt es **keine Lösung**, wie in den Hinweisen. Das Programm hat dafür **alle erreichbaren Zustände** der Reihe nach durchsucht und kein Ziel gefunden. Nur so kann man sicher sein, dass es keinen Plan gibt (der Suchraum ist hier klein genug, um ihn ganz zu durchsuchen).

**Beispiel 3** (`6 26 | 13 50`, Füllstände `0 0 | 0 20`, zusammen 20, jede Person soll 10 haben):

In [ ]:
loese('beispieldaten/fair-beispiel3.txt')

Behälter        6  26 |  13  50
Start           0   0 |   0  20
Schritt  1: 4 -> 1     6   0 |   0  14
Schritt  2: 1 -> 2     0   6 |   0  14
Schritt  3: 4 -> 1     6   6 |   0   8
Schritt  4: 1 -> 2     0  12 |   0   8
Schritt  5: 4 -> 1     6  12 |   0   2
Schritt  6: 1 -> 2     0  18 |   0   2
Schritt  7: 2 -> 3     0   5 |  13   2
Schritt  8: 3 -> 1     6   5 |   7   2
Schritt  9: 1 -> 2     0  11 |   7   2
Schritt 10: 3 -> 1     6  11 |   1   2
Schritt 11: 1 -> 2     0  17 |   1   2
Schritt 12: 3 -> 1     1  17 |   0   2
Schritt 13: 2 -> 3     1   4 |  13   2
Schritt 14: 3 -> 1     6   4 |   8   2
Ziel nach 14 Schritten: erste Person 10 , zweite Person 10 (319 Zustände besucht)


Der Plan hat **14 Schritte** und ist derselbe wie in den Hinweisen: Aus dem Behälter mit 20 Maß (vierter Behälter) wird abwechselnd 6 Maß in den ersten und dieser Inhalt in den zweiten Behälter gegossen, bis der zweite 18 enthält, dann geht es mit dem dritten Behälter (13) weiter. Am Ende hat die erste Person `6 + 4 = 10` Maß, die zweite `8 + 2 = 10`.

Es kann **mehrere gleich kurze Pläne** geben. Unser Programm findet den, den es zuerst erreicht. Welcher das ist, hängt von der Reihenfolge der Züge in `folgezustaende` ab.

**Weitere Beispiele aus den Lösungshinweisen** (hier von Hand eingegeben, nicht aus einer Datei):

In [ ]:
beispiele = [
    (2, [30, 28, 31, 27], [12, 9, 10, 13]),
    (2, [30, 28, 31, 27], [12, 9, 11, 12]),
    (4, [3, 5, 8, 13, 21, 34], [0, 0, 0, 0, 0, 34]),
]
for m, groesse, fuell in beispiele:
    plan, besucht = suche(m, groesse, fuell)
    if plan is None:
        print(groesse, fuell, '-> keine Lösung,', besucht, 'Zustände besucht')
    else:
        print(groesse, fuell, '->', len(plan), 'Schritte,', besucht, 'Zustände besucht')

[30, 28, 31, 27] [12, 9, 10, 13] -> 2 Schritte, 101 Zustände besucht
[30, 28, 31, 27] [12, 9, 11, 12] -> keine Lösung, 927 Zustände besucht
[3, 5, 8, 13, 21, 34] [0, 0, 0, 0, 0, 34] -> 6 Schritte, 2696 Zustände besucht


Die Ergebnisse (2 Schritte, keine Lösung, 6 Schritte) stimmen mit den Hinweisen überein. Zwei weitere Beispiele der Hinweise (sieben bzw. drei Behälter der Größe 10 oder 30 und einem fast leeren Behälter) haben viel mehr Zustände (Größenordnung `30^5` und mehr). Dafür bräuchte das Programm sehr viel Zeit und Speicher, sie sind hier bewusst weggelassen (nicht gerechnet).

## Wie groß wird der Suchraum?

Bei `k` Behältern und der Gesamtmenge `B` kann es sehr viele Zustände geben: alle Verteilungen von `B` Maß auf `k` Behälter, das sind höchstens `(B + k - 1) über (k - 1)`, wenn man die Fassungsvermögen nicht beachtet. Wir zählen die **besuchten** Zustände bei den Beispielen:

In [ ]:
for name in ['fair-beispiel1', 'fair-beispiel2', 'fair-beispiel3']:
    m, groesse, fuell = lies('beispieldaten/' + name + '.txt')
    plan, besucht = suche(m, groesse, fuell)
    print(name, ':', len(groesse), 'Behälter, Gesamtmenge', sum(fuell), '->', besucht, 'Zustände besucht')

fair-beispiel1 : 3 Behälter, Gesamtmenge 8 -> 16 Zustände besucht
fair-beispiel2 : 4 Behälter, Gesamtmenge 24 -> 138 Zustände besucht
fair-beispiel3 : 4 Behälter, Gesamtmenge 20 -> 319 Zustände besucht


Das Problem ist nach den Hinweisen **NP-schwer**: Aus dem **Partitionsproblem** (Zahlenmenge in zwei gleich schwere Hälften teilen) lässt sich ein Umfüllproblem bauen. Man nimmt für jede Zahl einen Behälter der ersten Person, der so groß ist wie die Summe aller Zahlen und mit der Zahl gefüllt ist, und für die zweite Person einen leeren Behälter derselben Größe. Weil jeder Behälter den **ganzen** Wein fassen kann, bleibt jeder Inhalt ungeteilt. Es geht also nur darum, welche Behälter man in den leeren gießt, bis dort die Hälfte liegt. Ein schnelles Verfahren für alle Eingaben ist deshalb nicht zu erwarten.

## Zum Schluss: Geht das mit MiniZinc?

Man kann den **Plan mit genau `T` Schritten** beschreiben:

- `z[t, i]`: Füllstand von Behälter `i` nach `t` Zügen. Zeile 0 ist der Start.
- `von[t]` und `nach[t]`: Der Zug `t` gießt von `von[t]` nach `nach[t]`. Dafür muss `von[t]` Wein haben und `nach[t]` Platz.
- Die gegossene `menge` ist das Minimum aus Inhalt des Quellbehälters und freiem Platz im Zielbehälter. Die `let`-Anweisung gibt dieser Hilfsgröße einen Namen. `bool2int(i = von[t])` ist 1 für den Quellbehälter und sonst 0.
- Am Ende hat die erste Person genau die Hälfte.

Bei `solve satisfy` sucht der Solver irgendeinen Plan mit **genau** `T` Zügen. Die **minimale** Zahl bekommt man, indem man `T = 1, 2, 3, ...` nacheinander probiert: Der erste Wert mit Lösung ist das Minimum.

**Modell** (Datei `umfuellen.mzn`):

```minizinc
int: T;                                    % Zahl der Züge
int: k;                                    % Zahl der Behälter
int: m;                                    % die ersten m Behälter gehören der ersten Person
array[1..k] of int: g;                     % Fassungsvermögen
array[1..k] of int: b0;                    % Füllstände am Anfang
array[0..T, 1..k] of var 0..max(g): z;     % z[t, i]: Füllstand von Behälter i nach t Zügen
array[1..T] of var 1..k: von;
array[1..T] of var 1..k: nach;

constraint forall(i in 1..k)(z[0, i] = b0[i]);
constraint forall(t in 1..T)(
  von[t] != nach[t] /\ z[t-1, von[t]] > 0 /\ z[t-1, nach[t]] < g[nach[t]] /\
  let { var int: menge = min(z[t-1, von[t]], g[nach[t]] - z[t-1, nach[t]]) } in
  forall(i in 1..k)(z[t, i] = z[t-1, i] - bool2int(i = von[t]) * menge + bool2int(i = nach[t]) * menge));
constraint 2 * sum(i in 1..m)(z[T, i]) = sum(b0);

solve satisfy;
output [show(von), "\n", show(nach), "\n"];
```

**Daten** für Beispiel 1 mit `T = 7` (`umfuellen.dzn`, in Python aus der Textdatei erzeugt):

```
T = 7;
k = 3;
m = 1;
g = [8, 5, 3];
b0 = [8, 0, 0];
```

**Ausgabe** (Gecode, sofort):

```
[1, 2, 3, 2, 1, 2, 3]
[2, 3, 1, 3, 2, 3, 1]
----------
```

Das ist ein Plan mit 7 Zügen (`von` und `nach`, Behälter ab 1), also dasselbe Ergebnis wie in Python. Mit `T = 1` bis `6` meldet MiniZinc bei Beispiel 1 jeweils `=====UNSATISFIABLE=====`, **7 ist also das Minimum**.

**Grenzen** (alles gemessen, Zeitlimit jeweils 20 Sekunden):

- **Beispiel 2** hat keine Lösung. MiniZinc kann das aber nur für **ein festes `T`** zeigen: Mit Chuffed ist `T = 1` bis `8` unerfüllbar (bei `T = 6` etwa 2 Sekunden, bei 7 und 8 schon über 10 Sekunden), ab `T = 9` wird der Solver nicht mehr fertig. „Es gibt **nie** einen Plan“ lässt sich so nicht beweisen, denn man müsste jedes `T` ausschließen. Die Breitensuche in Python prüft stattdessen alle 138 erreichbaren Zustände und ist damit fertig.
- **Beispiel 3** hat einen Plan mit 14 Zügen. Bei `T = 12` bis `15` wird weder Gecode noch Chuffed in 20 Sekunden fertig, auch bei `T = 14` nicht, wo es eine Lösung gibt.

Das Problem ist also für MiniZinc **nur bei ganz kleinen Beispielen** geeignet. Die Breitensuche ist hier viel besser: Sie zählt nicht über die Zugzahl `T`, sondern besucht jeden Zustand genau einmal.

## Für Neugierige

- **Ungerade Gesamtmenge:** Dann ist die Aufgabe sofort unlösbar. Das Programm stellt es auch durch Suchen fest, mit einem Test vorab spart man sich aber die Suche (`sum(fuell) % 2 == 1`).
- **Symmetrie:** Mit `(f_1, ..., f_k)` ist auch `(G_1 - f_1, ..., G_k - f_k)` erreichbar, wenn man den Weg umkehrt. Die Hinweise nennen das bei Abbildung 1. Man kann damit die Suche von beiden Seiten gleichzeitig führen (**bidirektionale Suche**) und spart bei großen Beispielen viele Zustände.
- **Weniger Speicher:** Statt jeden Zustand als Tupel abzulegen, kann man ihn als Zahl kodieren.
- **Große Beispiele:** Die Hinweise zeigen Beispiele mit hoher Laufzeit. Mit der Zahl der Zustände (bis `(B + k - 1) über (k - 1)`) wächst der Aufwand schnell. Dass es ein schnelles Verfahren für alle Eingaben gibt, ist nach der Reduktion vom Partitionsproblem unwahrscheinlich.